# MOXEC — Complete End-to-End Experiment (Single Dataset)

**Multi-Objective Automated Machine Learning for Explainable and Efficient Classification**

This notebook runs the **entire MOXEC pipeline on one dataset**, start to finish:

1. Data loading and leakage-safe preprocessing
2. Search space definition (9 model families)
3. Three objectives: **f1 = MCC**, **f2 = SHAP faithfulness (Φ)**, **f3 = structural inference-cost proxy**
4. NSGA-II multi-objective CASH search (Optuna), checkpointed and resumable
5. Pareto front extraction, visualization, and navigation (knee point / cost-constrained / weighted MES)
6. Baselines: FLAML, AutoGluon (optional), Random Search, single-objective TPE
7. Evaluation: hypervolume (pymoo, exact), per-objective breakdown, Wilcoxon across seeds
8. Saved artifacts for later aggregation across the remaining 11 datasets

**Default dataset:** Diabetic Retinopathy Debrecen (UCI id 329) — binary, medical, 1151×19, no missing values.
To reuse this notebook for another dataset, change only the `CONFIG` cell below — every downstream
cell reads from `CONFIG` and adapts automatically (binary vs. multiclass MCC/SHAP handling included).

**Runtime:** ~20–40 minutes on Colab/Kaggle free-tier CPU for the default dataset at `N_TRIALS=100`, `N_SEEDS=3`.

---


## 0. Install dependencies

Run once per runtime. Safe to re-run.

In [23]:
# Core dependencies for the full pipeline (data, models, search, faithfulness, stats, plotting)
#
# --break-system-packages is required on Homebrew/Debian Python (PEP 668 "externally
# managed environment") when installing into the system interpreter rather than a venv.
# It is a no-op and harmless on Colab/Kaggle, where every runtime is disposable anyway --
# so this single install line works unmodified in all three environments.
%pip install -q --break-system-packages ucimlrepo optuna shap imbalanced-learn xgboost lightgbm flaml scikit-posthocs pymoo pyarrow

# scikit-learn / pandas / numpy / scipy / matplotlib ship preinstalled on Colab and Kaggle.
# If running somewhere minimal, uncomment:
# %pip install -q --break-system-packages scikit-learn pandas numpy scipy matplotlib

print("Core install complete.")
print("If you saw an 'externally-managed-environment' error above, the install did NOT")
print("succeed despite this message -- re-run this cell; --break-system-packages should")
print("resolve it. If you still see it, your pip is older than the flag: run")
print("  python3 -m pip install --upgrade pip")
print("in Terminal first, then re-run this cell.")


Note: you may need to restart the kernel to use updated packages.
Core install complete.
If you saw an 'externally-managed-environment' error above, the install did NOT
succeed despite this message -- re-run this cell; --break-system-packages should
resolve it. If you still see it, your pip is older than the flag: run
  python3 -m pip install --upgrade pip
in Terminal first, then re-run this cell.


In [ ]:
# OPTIONAL — AutoGluon baseline. Heavy (~2-4 min install, large dependency tree).
# Skip this cell if you don't want the AutoGluon baseline; the notebook detects its
# absence automatically and simply omits it from the comparison.
RUN_AUTOGLUON = True  # set False to skip installing/running AutoGluon entirely

if RUN_AUTOGLUON:
    %pip install -q --break-system-packages "autogluon.tabular[all]"
    import importlib
    for pkg in ["torch", "catboost", "fastai"]:
        try:
            mod = importlib.import_module(pkg)
            print(f"  {pkg}: OK (version {getattr(mod, '__version__', 'unknown')})")
        except ImportError as e:
            print(f"  {pkg}: STILL MISSING -- {e}")
    print("AutoGluon install complete.")
else:
    print("Skipping AutoGluon install (RUN_AUTOGLUON=False).")


Note: you may need to restart the kernel to use updated packages.
AutoGluon installed.


## 1. Imports and configuration

In [25]:
import warnings
warnings.filterwarnings("ignore")

import os, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (enables 3D projection)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import matthews_corrcoef, f1_score, average_precision_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb

import shap
import optuna
from optuna.samplers import NSGAIISampler, RandomSampler, TPESampler

from pymoo.indicators.hv import HV
from scipy.stats import wilcoxon

optuna.logging.set_verbosity(optuna.logging.WARNING)
np.seterr(all="ignore")

print("Imports OK.")


Imports OK.


In [26]:
# ============================================================
# CONFIG — the only cell you need to change to re-run this notebook
# on a different dataset from the MOXEC portfolio.
# ============================================================
CONFIG = {
    "uci_id": 602,                  # Dry Bean (agriculture, 7-class, 13611x16)
    "dataset_name": "dry_bean",
    "task_type": "multiclass",      # "binary" or "multiclass" — controls MCC/SHAP handling below
    "n_trials": 100,               # NSGA-II / Random / TPE trial budget
    "n_seeds": 3,                  # independent repeats for statistical comparison
    "cv_folds": 5,
    "faithfulness_sample_size": 200,   # capped by test-fold size automatically
    "faithfulness_steps": 8,           # log/linear-spaced deletion/insertion steps
    "output_dir": "./moxec_results",
    "use_drive_for_colab": True,   # if running on Colab, persist Optuna DB + results to Drive
}

# --- detect environment & set persistent storage path -------------------------------------
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IN_COLAB and CONFIG["use_drive_for_colab"]:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path(f"/content/drive/MyDrive/MOXEC/{CONFIG['dataset_name']}")
else:
    BASE_DIR = Path(CONFIG["output_dir"]) / CONFIG["dataset_name"]

BASE_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB = f"sqlite:///{BASE_DIR / 'optuna_studies.db'}"
print(f"Environment: {'Colab' if IN_COLAB else 'local/Kaggle'}")
print(f"Artifacts will be saved under: {BASE_DIR.resolve()}")
print(f"Optuna storage: {STUDY_DB}")


Environment: local/Kaggle
Artifacts will be saved under: /Users/gyanendrachaubey/Documents/MOXEC/moxec_results/dry_bean
Optuna storage: sqlite:///moxec_results/dry_bean/optuna_studies.db


## 2. Data loading

Fetched from the UCI Machine Learning Repository via `ucimlrepo`. If the API endpoint is
unreachable (rare, but happens on some networks), a direct-download fallback is used.

**Verify the printed shape and class balance against the UCI listing** before trusting downstream
results — this is a cheap check that catches a wrong `uci_id` immediately.

In [27]:
from ucimlrepo import fetch_ucirepo

def load_uci_dataset(uci_id):
    ds = fetch_ucirepo(id=uci_id)
    X = ds.data.features.copy()
    y = ds.data.targets.copy()
    # targets sometimes come back as a 1-column DataFrame — squeeze to Series
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    return X, y, ds.metadata

X_raw, y_raw, meta = load_uci_dataset(CONFIG["uci_id"])

print(f"Dataset: {meta.name}")
print(f"Shape:   {X_raw.shape[0]} instances x {X_raw.shape[1]} features")
print(f"Target distribution:\n{y_raw.value_counts()}")
print(f"\nMissing values per column (top 5):\n{X_raw.isna().sum().sort_values(ascending=False).head(5)}")


Dataset: Dry Bean
Shape:   13611 instances x 16 features
Target distribution:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

Missing values per column (top 5):
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRatio        0
dtype: int64


In [28]:
# Encode target to 0..K-1 integer labels (required by sklearn/xgboost/lightgbm)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y_raw)
n_classes = len(le.classes_)
assert (n_classes == 2 and CONFIG["task_type"] == "binary") or (n_classes > 2 and CONFIG["task_type"] == "multiclass"), \
    f"CONFIG['task_type']={CONFIG['task_type']!r} does not match detected {n_classes} classes — fix CONFIG."

# Identify categorical vs numeric columns generically (works across the 12-dataset portfolio)
cat_cols = X_raw.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
print(f"Classes: {list(le.classes_)}  ->  encoded as {sorted(set(y))}")
print(f"Categorical columns: {cat_cols if cat_cols else 'none'}")
print(f"Numeric columns: {len(num_cols)}")

# One-hot encode any categoricals up front (kept outside the CV loop deliberately —
# this is a fixed structural transform, not a fitted statistic, so it carries no leakage risk)
if cat_cols:
    X_enc = pd.get_dummies(X_raw, columns=cat_cols, dummy_na=True)
else:
    X_enc = X_raw.copy()

# Median-impute any missing numeric values using a placeholder now; the REAL per-fold
# imputation happens inside the pipeline below. This top-level fill only prevents
# downstream sklearn errors on columns imblearn/sklearn transformers can't handle as NaN
# for transformers that don't support NaN natively (SMOTE, MLP). Simple mean fill here
# is a coarse placeholder — for M11-style heavy-missingness datasets, replace with
# indicator + in-fold imputation (see PALE_lean_protocol.md Preprocessing note).
X_enc = X_enc.fillna(X_enc.median(numeric_only=True))

# Guard against duplicate column names in the raw UCI data. Some UCI datasets ship
# non-unique feature names (repeated measurement labels at different scales, generic
# "0","1",... headers, etc.) -- harmless for the numpy-array path used throughout this
# notebook, but AutoGluon's TabularPredictor requires unique DataFrame columns and fails
# with an opaque "Column names are not unique" error if this isn't handled here.
def _dedupe_columns(cols):
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__{seen[c]}")
    return out

if X_enc.columns.duplicated().any():
    dup_count = int(X_enc.columns.duplicated().sum())
    print(f"WARNING: {dup_count} duplicate column name(s) in the raw data -- renaming "
          f"to make them unique (needed for AutoGluon; harmless for everything else).")
    X_enc.columns = _dedupe_columns(X_enc.columns.tolist())

X = X_enc.values.astype(np.float64)
feature_names = X_enc.columns.tolist()
print(f"\nFinal design matrix: X.shape = {X.shape}")


Classes: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']  ->  encoded as [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Categorical columns: none
Numeric columns: 16

Final design matrix: X.shape = (13611, 16)


## 3. Leakage-safe preprocessing pipeline

All resampling and scaling is fit **inside each training fold only**, via `imblearn.Pipeline`.
This is the single most important correctness rule in the whole protocol — SMOTE (or any
resampler) fit before the split leaks test-set structure into training and inflates every
downstream number. Audit any code you did **not** write this way before trusting its results.

In [29]:
def build_preprocessing_steps(trial, n_features):
    '''Suggest a leakage-safe preprocessing configuration for this trial.
    Returns a list of (name, transformer) tuples for imblearn.Pipeline.'''
    steps = []

    scaler_choice = trial.suggest_categorical("scaler", ["none", "standard", "minmax", "robust"])
    if scaler_choice == "standard":
        steps.append(("scaler", StandardScaler()))
    elif scaler_choice == "minmax":
        steps.append(("scaler", MinMaxScaler()))
    elif scaler_choice == "robust":
        steps.append(("scaler", RobustScaler()))

    fs_choice = trial.suggest_categorical("feature_selection", ["none", "kbest"])
    if fs_choice == "kbest":
        k_frac = trial.suggest_float("kbest_frac", 0.5, 1.0)
        k = max(1, int(round(k_frac * n_features)))
        steps.append(("select", SelectKBest(score_func=mutual_info_classif, k=k)))

    imb_choice = trial.suggest_categorical("imbalance", ["none", "smote"])
    if imb_choice == "smote":
        steps.append(("smote", SMOTE(random_state=0)))

    return steps


## 4. Search space — 9 model families

Deliberately spans the explainability spectrum: a linear model, a single tree, two bagged
tree ensembles, two boosted ensembles, an instance-based method, a probabilistic model, and
one neural network. Trees are expected to dominate the Pareto front on tabular data — the
point of including MLP is to show that quantitatively, not to expect it to win.

In [30]:
# IMPORTANT: Optuna categorical choices persisted to SQLite/RDB storage must be plain
# str/int/float/bool/None -- NOT tuples or lists. Optuna itself warns about this at
# suggestion time ("Choices for a categorical distribution should be a tuple of None,
# bool, int, float and str for persistent storage"), but the warning is easy to miss and
# the failure only surfaces later, asymmetrically: a tuple like (64, 32) round-trips
# through SQLite's JSON encoding as a *list* [64, 32], and list != tuple in Python, so any
# sampler that needs to reconstruct past trials' parameter values against the original
# choices (TPESampler building its probability model) throws
#     ValueError: '[64, 32]' not in ((32,), (64,), (64, 32), (128, 64))
# RandomSampler and NSGA-II can run for a while without hitting this (they don't always
# need to compare historical values against the distribution), which is exactly what
# makes it a landmine: it can crash a resumed run hours in rather than immediately.
# Fix: suggest a STRING key and map it to the real tuple locally -- the persisted value
# is then always a str, which survives the round trip.
MLP_ARCHS = {"32": (32,), "64": (64,), "64_32": (64, 32), "128_64": (128, 64)}


def suggest_model(trial, n_classes, random_state=0):
    '''Suggest a model family + its hyperparameters. Returns (name, estimator).'''
    family = trial.suggest_categorical(
        "model_family",
        ["logreg", "dtree", "rforest", "extratrees", "xgboost", "lightgbm", "knn", "nb", "mlp"],
    )

    if family == "logreg":
        C = trial.suggest_float("lr_C", 1e-3, 1e2, log=True)
        penalty = trial.suggest_categorical("lr_penalty", ["l1", "l2"])
        model = LogisticRegression(C=C, penalty=penalty, solver="liblinear",
                                    max_iter=2000, random_state=random_state)

    elif family == "dtree":
        depth = trial.suggest_int("dt_max_depth", 2, 20)
        min_split = trial.suggest_int("dt_min_samples_split", 2, 20)
        model = DecisionTreeClassifier(max_depth=depth, min_samples_split=min_split,
                                        random_state=random_state)

    elif family == "rforest":
        n_est = trial.suggest_int("rf_n_estimators", 20, 300)
        depth = trial.suggest_int("rf_max_depth", 3, 20)
        model = RandomForestClassifier(n_estimators=n_est, max_depth=depth,
                                        n_jobs=-1, random_state=random_state)

    elif family == "extratrees":
        n_est = trial.suggest_int("et_n_estimators", 20, 300)
        depth = trial.suggest_int("et_max_depth", 3, 20)
        model = ExtraTreesClassifier(n_estimators=n_est, max_depth=depth,
                                      n_jobs=-1, random_state=random_state)

    elif family == "xgboost":
        n_est = trial.suggest_int("xgb_n_estimators", 20, 300)
        depth = trial.suggest_int("xgb_max_depth", 2, 12)
        lr = trial.suggest_float("xgb_lr", 1e-3, 0.5, log=True)
        objective = "binary:logistic" if n_classes == 2 else "multi:softprob"
        kwargs = {} if n_classes == 2 else {"num_class": n_classes}
        model = xgb.XGBClassifier(n_estimators=n_est, max_depth=depth, learning_rate=lr,
                                   objective=objective, eval_metric="logloss",
                                   n_jobs=-1, random_state=random_state, **kwargs)

    elif family == "lightgbm":
        n_est = trial.suggest_int("lgb_n_estimators", 20, 300)
        leaves = trial.suggest_int("lgb_num_leaves", 7, 127)
        lr = trial.suggest_float("lgb_lr", 1e-3, 0.5, log=True)
        model = lgb.LGBMClassifier(n_estimators=n_est, num_leaves=leaves, learning_rate=lr,
                                    n_jobs=-1, random_state=random_state, verbosity=-1)

    elif family == "knn":
        k = trial.suggest_int("knn_k", 3, 31)
        model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)

    elif family == "nb":
        model = GaussianNB()

    elif family == "mlp":
        arch_key = trial.suggest_categorical("mlp_units", list(MLP_ARCHS.keys()))
        n_units = MLP_ARCHS[arch_key]
        alpha = trial.suggest_float("mlp_alpha", 1e-5, 1e-1, log=True)
        model = MLPClassifier(hidden_layer_sizes=n_units, alpha=alpha,
                               max_iter=500, random_state=random_state)

    return family, model


## 5. Objective functions

- **f1 — MCC** (maximize): robust under class imbalance, unlike accuracy.
- **f2 — Φ faithfulness** (maximize): SHAP-ranked deletion/insertion, tracked **per instance
  on the model's own predicted class** and averaged as AUCs (the RISE protocol) — averaging
  raw probabilities across instances first, as a naive implementation would, cancels signal
  across classes and silently produces meaningless values. This was verified against a
  synthetic sanity check before being used here.
- **f3 — structural cost proxy** (minimize): hardware-independent, reproducible on any machine,
  validated below against measured wall-clock latency.

In [31]:
def compute_mcc(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return matthews_corrcoef(y_te, pred), model


def structural_cost(model, family, n_train, n_features):
    '''Hardware-independent proxy for inference cost. Higher = more expensive.'''
    if family in ("rforest", "extratrees"):
        return float(sum(est.tree_.node_count for est in model.estimators_))
    if family == "dtree":
        return float(model.tree_.node_count)
    if family == "xgboost":
        booster = model.get_booster()
        df = booster.trees_to_dataframe()
        return float(len(df))  # total nodes across all boosted trees
    if family == "lightgbm":
        return float(model.booster_.num_trees() * (2 ** 6))  # trees x approx nodes/tree at default depth
    if family == "logreg":
        return float(np.sum(np.abs(model.coef_) > 1e-8))
    if family == "knn":
        return float(n_train * n_features)  # lazy learner — cost scales with training set
    if family == "nb":
        return float(n_features)
    if family == "mlp":
        return float(sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_))
    raise ValueError(f"Unknown family: {family}")


def get_shap_values(model, family, X_background, X_explain, n_classes):
    '''Dispatch to the fastest correct SHAP explainer per model family.'''
    tree_families = {"dtree", "rforest", "extratrees", "xgboost", "lightgbm"}
    if family in tree_families:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_explain, check_additivity=False)
    elif family == "logreg":
        explainer = shap.LinearExplainer(model, X_background)
        sv = explainer.shap_values(X_explain)
    else:
        # KernelSHAP fallback for KNN / NB / MLP -- capped budget to stay tractable
        bg = shap.kmeans(X_background, min(30, len(X_background)))
        predict_fn = model.predict_proba
        explainer = shap.KernelExplainer(predict_fn, bg)
        sv = explainer.shap_values(X_explain, nsamples=100, silent=True)

    # Normalize output shape to (n_classes, n_instances, n_features). SHAP's return shape
    # is inconsistent across explainers/model families/versions:
    #   - list of per-class arrays  -> stack directly
    #   - (n, d, n_classes)         -> RF/DT/KernelExplainer on multi-output models
    #   - (n, d)                   -> XGBoost/LightGBM/LinearExplainer on BINARY problems:
    #                                  a single array for class 1 only.
    # That last case is a real pitfall: naively wrapping it as shape (1, n, d) and then
    # indexing by predicted class (0 or 1) IndexErrors whenever the predicted class is 0.
    # The fix is to expand it into both classes: class-1 attribution is the returned
    # array, class-0 attribution is its negation (the two classes' probabilities sum to
    # 1, so their local attributions are equal and opposite).
    if isinstance(sv, list):
        return np.array(sv)                  # (n_classes, n, d)
    sv = np.asarray(sv)
    if sv.ndim == 3:
        return np.transpose(sv, (2, 0, 1))    # (n, d, n_classes) -> (n_classes, n, d)
    if n_classes == 2:
        return np.stack([-sv, sv])            # (2, n, d)
    raise ValueError(f"Unexpected SHAP output shape {sv.shape} for a {n_classes}-class problem")


def faithfulness(model, family, X_background, X_explain, k_steps=8):
    '''Φ = InsertionAUC - DeletionAUC, per-instance on the predicted class, averaged.
    See notebook markdown above for why this must be per-instance, not pooled.'''
    n, d = X_explain.shape
    background_vec = np.median(X_background, axis=0)
    proba_full = model.predict_proba(X_explain)
    target_cls = proba_full.argmax(axis=1)
    n_classes_local = proba_full.shape[1]

    sv_all = get_shap_values(model, family, X_background, X_explain, n_classes_local)  # (C, n, d)
    # select each instance's SHAP vector for ITS OWN predicted class
    sv = np.stack([sv_all[target_cls[i], i, :] for i in range(n)])  # (n, d)

    order = np.argsort(-np.abs(sv), axis=1)
    steps = np.unique(np.linspace(0, d, k_steps, dtype=int))
    x = steps / d

    del_curves = np.zeros((n, len(steps)))
    ins_curves = np.zeros((n, len(steps)))
    for si, k in enumerate(steps):
        Xdel = X_explain.copy()
        Xins = np.tile(background_vec, (n, 1))
        for i in range(n):
            idx = order[i, :k]
            Xdel[i, idx] = background_vec[idx]
            Xins[i, idx] = X_explain[i, idx]
        del_curves[:, si] = model.predict_proba(Xdel)[np.arange(n), target_cls]
        ins_curves[:, si] = model.predict_proba(Xins)[np.arange(n), target_cls]

    del_auc = np.array([np.trapezoid(del_curves[i], x) for i in range(n)])
    ins_auc = np.array([np.trapezoid(ins_curves[i], x) for i in range(n)])
    phi = float((ins_auc - del_auc).mean())
    return phi, float(del_auc.mean()), float(ins_auc.mean())


def measured_latency_ms(model, X_sample, n_warmup=20, n_repeats=100):
    '''Single-sample (batch=1) wall-clock latency, median over repeats. Used only to
    VALIDATE the structural proxy, never as the optimized objective (Colab timing is noisy).'''
    x1 = X_sample[0:1]
    for _ in range(n_warmup):
        model.predict(x1)
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        model.predict(x1)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))

print("Objective functions defined.")


Objective functions defined.


### 5a. Faithfulness sanity check (do this before trusting Φ downstream)

Compares Φ computed on a small sample with few steps against a larger sample with more
steps, on a handful of pilot configurations. If the rank correlation is low, increase
`faithfulness_sample_size` / `faithfulness_steps` in `CONFIG` before running the full search —
otherwise you are optimizing noise.

In [ ]:
from scipy.stats import spearmanr

def pilot_faithfulness_check(X, y, n_configs=8, seed=0):
    rng = np.random.default_rng(seed)
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    X_tr, X_te, y_tr, y_te = X[tr_idx], y[tr_idx], y[tr_idx], y[te_idx]  # placeholder overwritten below
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    small_n = min(int(CONFIG["faithfulness_sample_size"] * 0.5), len(X_te))
    large_n = min(len(X_te), max(small_n * 2, small_n + 1))

    phi_small, phi_large = [], []
    study_pilot = optuna.create_study(directions=["maximize"], sampler=RandomSampler(seed=seed))

    for i in range(n_configs):
        trial = study_pilot.ask()
        family, model = suggest_model(trial, n_classes, random_state=seed)
        try:
            model.fit(X_tr, y_tr)
        except Exception:
            study_pilot.tell(trial, 0.0)
            continue
        idx_small = rng.choice(len(X_te), size=small_n, replace=False)
        idx_large = rng.choice(len(X_te), size=large_n, replace=False)
        try:
            p_s, _, _ = faithfulness(model, family, X_tr, X_te[idx_small], k_steps=max(4, CONFIG["faithfulness_steps"] // 2))
            p_l, _, _ = faithfulness(model, family, X_tr, X_te[idx_large], k_steps=CONFIG["faithfulness_steps"] * 2)
        except Exception as e:
            study_pilot.tell(trial, 0.0)
            continue
        phi_small.append(p_s); phi_large.append(p_l)
        study_pilot.tell(trial, 0.0)

    if len(phi_small) < 3:
        print("Too few successful pilot configs to compute correlation — inspect exceptions above.")
        return None, None, len(phi_small)
    rho, pval = spearmanr(phi_small, phi_large)
    print(f"Pilot configs evaluated: {len(phi_small)}")
    print(f"Spearman correlation (reduced-budget Φ vs larger-budget Φ): rho={rho:.3f}  p={pval:.4f}")
    if rho < 0.80:
        print("WARNING: correlation below 0.80 — increase faithfulness_sample_size/steps in CONFIG.")
    else:
        print("OK: reduced-budget faithfulness estimate is adequately correlated with the larger-budget estimate.")
    return rho, pval, len(phi_small)

faithfulness_sanity_rho, faithfulness_sanity_pval, faithfulness_sanity_n_configs = pilot_faithfulness_check(X, y, n_configs=8, seed=0)

## 6. NSGA-II multi-objective CASH search

Wires preprocessing + model selection + all three objectives into one Optuna study using
`NSGAIISampler`. Persisted to SQLite so a disconnected Colab/Kaggle session can resume exactly
where it left off — re-run this cell after a disconnect and it will pick up remaining trials.

In [33]:
def make_objective(X, y, seed):
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)

    def objective(trial):
        pre_steps = build_preprocessing_steps(trial, X.shape[1])
        family, model = suggest_model(trial, n_classes, random_state=seed)

        mccs, phis, costs = [], [], []
        for tr_idx, te_idx in skf.split(X, y):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            pipe = ImbPipeline(pre_steps + [("clf", model)]) if pre_steps else ImbPipeline([("clf", model)])
            try:
                pipe.fit(X_tr, y_tr)
            except Exception:
                raise optuna.TrialPruned()

            pred = pipe.predict(X_te)
            mccs.append(matthews_corrcoef(y_te, pred))

            fitted_model = pipe.named_steps["clf"]
            # transform X_te through preprocessing (minus resampler, which only applies at fit time)
            X_te_pre = X_te
            for name, step in pipe.steps[:-1]:
                if name != "smote":
                    X_te_pre = step.transform(X_te_pre)
            X_tr_pre = X_tr
            for name, step in pipe.steps[:-1]:
                if name != "smote":
                    X_tr_pre = step.transform(X_tr_pre) if not hasattr(step, "fit_resample") else X_tr_pre

            n_faith = min(CONFIG["faithfulness_sample_size"], len(X_te_pre))
            idx = np.random.RandomState(seed).choice(len(X_te_pre), size=n_faith, replace=False)
            try:
                phi, _, _ = faithfulness(fitted_model, family, X_tr_pre, X_te_pre[idx],
                                          k_steps=CONFIG["faithfulness_steps"])
            except Exception:
                phi = 0.0
            phis.append(phi)

            costs.append(structural_cost(fitted_model, family, len(X_tr), X_tr_pre.shape[1]))

        trial.set_user_attr("model_family", family)
        return float(np.mean(mccs)), float(np.mean(phis)), float(np.mean(costs))

    return objective


def run_nsga2_search(X, y, seed, n_trials, study_name_suffix=""):
    study = optuna.create_study(
        study_name=f"moxec_nsga2_seed{seed}{study_name_suffix}",
        directions=["maximize", "maximize", "minimize"],
        sampler=NSGAIISampler(seed=seed),
        storage=STUDY_DB,
        load_if_exists=True,
    )
    remaining = max(0, n_trials - len(study.trials))
    if remaining > 0:
        print(f"[seed {seed}] running {remaining} new trials ({len(study.trials)} already done)...")
        study.optimize(make_objective(X, y, seed), n_trials=remaining, show_progress_bar=False)
    else:
        print(f"[seed {seed}] already has {len(study.trials)} trials — nothing to do (resumed).")
    return study

print("NSGA-II search functions defined. Run the next cell to launch the search.")


NSGA-II search functions defined. Run the next cell to launch the search.


In [34]:
# This is the long-running cell. Safe to interrupt and re-run — Optuna resumes from the DB.
nsga2_studies = {}
for seed in range(CONFIG["n_seeds"]):
    nsga2_studies[seed] = run_nsga2_search(X, y, seed, CONFIG["n_trials"])

print("\nNSGA-II search complete for all seeds.")
for seed, study in nsga2_studies.items():
    print(f"  seed {seed}: {len(study.trials)} trials, {len(study.best_trials)} on the Pareto front")


[seed 0] running 100 new trials (0 already done)...
[seed 1] running 100 new trials (0 already done)...
[seed 2] running 100 new trials (0 already done)...

NSGA-II search complete for all seeds.
  seed 0: 100 trials, 21 on the Pareto front
  seed 1: 100 trials, 17 on the Pareto front
  seed 2: 100 trials, 19 on the Pareto front


## 7. Pareto front extraction and visualization

In [35]:
def trials_to_dataframe(study):
    '''Handles both the 3-objective MOXEC/random studies and the single-objective
    TPE baseline study -- the latter\'s trials carry only one value (MCC), and
    indexing t.values[1]/[2] unconditionally IndexErrors on it.'''
    rows = []
    for t in study.trials:
        if t.values is None:
            continue
        row = {
            "trial": t.number,
            "model_family": t.user_attrs.get("model_family", "unknown"),
            **{f"param_{k}": v for k, v in t.params.items()},
        }
        if len(t.values) == 3:
            row["mcc"], row["phi"], row["cost"] = t.values
        elif len(t.values) == 1:
            row["mcc"] = t.values[0]
        else:
            raise ValueError(f"Unexpected number of objective values: {len(t.values)}")
        rows.append(row)
    return pd.DataFrame(rows)

def pareto_front_df(study):
    df = trials_to_dataframe(study)
    front_numbers = {t.number for t in study.best_trials}
    return df[df["trial"].isin(front_numbers)]

seed0_df = trials_to_dataframe(nsga2_studies[0])
seed0_front = pareto_front_df(nsga2_studies[0])
print(f"Seed 0: {len(seed0_df)} total trials, {len(seed0_front)} on the Pareto front")
seed0_front[["trial", "mcc", "phi", "cost", "model_family"]].sort_values("mcc", ascending=False).head(10)


Seed 0: 91 total trials, 21 on the Pareto front


,trial,mcc,phi,cost,model_family
17,21,0.918380,0.543432,775.0,mlp
22,26,0.917614,0.556922,3399.0,mlp
74,83,0.915497,0.546325,775.0,mlp
66,75,0.904582,0.547392,3079.0,mlp
11,12,0.893059,0.371205,551.0,mlp
10,10,0.878771,0.411226,698.2,dtree
77,86,0.877157,0.454762,771.0,dtree
43,49,0.875896,0.334575,16.0,nb
45,51,0.875719,0.484134,110.6,dtree
53,61,0.875094,0.339976,16.0,nb


In [ ]:
def plot_pareto_front(seed):
    '''Build and save the 3-panel Pareto front figure for one seed. Previously this
    only ran for seed 0 -- fine for a quick look, but it silently meant seeds 1/2 had
    no visual record on disk even though their trial/front data was saved (§12).'''
    df_s = trials_to_dataframe(nsga2_studies[seed])
    front_s = pareto_front_df(nsga2_studies[seed])

    fig = plt.figure(figsize=(15, 5))
    families = df_s["model_family"].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(families)))
    family_color = dict(zip(families, colors))

    ax1 = fig.add_subplot(1, 3, 1, projection="3d")
    for fam in families:
        sub = df_s[df_s["model_family"] == fam]
        ax1.scatter(sub["mcc"], sub["phi"], sub["cost"], label=fam, alpha=0.5, s=20, color=family_color[fam])
    ax1.scatter(front_s["mcc"], front_s["phi"], front_s["cost"],
                color="black", marker="*", s=140, label="Pareto front", edgecolor="white", linewidth=0.5)
    ax1.set_xlabel("MCC"); ax1.set_ylabel("Faithfulness (Φ)"); ax1.set_zlabel("Cost proxy")
    ax1.set_title(f"3D objective space (seed {seed})")

    ax2 = fig.add_subplot(1, 3, 2)
    for fam in families:
        sub = df_s[df_s["model_family"] == fam]
        ax2.scatter(sub["mcc"], sub["phi"], alpha=0.5, s=20, color=family_color[fam], label=fam)
    ax2.scatter(front_s["mcc"], front_s["phi"], color="black", marker="*", s=140, edgecolor="white", linewidth=0.5)
    ax2.set_xlabel("MCC"); ax2.set_ylabel("Faithfulness (Φ)"); ax2.set_title(f"MCC vs. Faithfulness (seed {seed})")

    ax3 = fig.add_subplot(1, 3, 3)
    for fam in families:
        sub = df_s[df_s["model_family"] == fam]
        ax3.scatter(sub["mcc"], sub["cost"], alpha=0.5, s=20, color=family_color[fam], label=fam)
    ax3.scatter(front_s["mcc"], front_s["cost"], color="black", marker="*", s=140, edgecolor="white", linewidth=0.5)
    ax3.set_xlabel("MCC"); ax3.set_ylabel("Cost proxy"); ax3.set_title(f"MCC vs. Cost (seed {seed})")
    ax3.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)

    plt.tight_layout()
    out_path = BASE_DIR / f"pareto_front_seed{seed}.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
    return out_path


pareto_plot_paths = [plot_pareto_front(seed) for seed in range(CONFIG["n_seeds"])]

## 8. Latency proxy validation

The structural proxy is what's optimized (deterministic, reproducible). Here we check it
actually correlates with measured single-sample wall-clock latency on a sample of Pareto-front
configurations — the honest justification for using a proxy at all.

In [ ]:
def refit_trial_pipeline(trial_params, X_tr, y_tr, seed):
    '''Rebuild and refit the exact pipeline a given Optuna trial specified.'''
    fixed_trial = optuna.trial.FixedTrial(trial_params)
    pre_steps = build_preprocessing_steps(fixed_trial, X_tr.shape[1])
    family, model = suggest_model(fixed_trial, n_classes, random_state=seed)
    pipe = ImbPipeline(pre_steps + [("clf", model)]) if pre_steps else ImbPipeline([("clf", model)])
    pipe.fit(X_tr, y_tr)
    return family, pipe

skf_val = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=0)
tr_idx, te_idx = next(skf_val.split(X, y))
X_tr_v, X_te_v = X[tr_idx], X[te_idx]
y_tr_v, y_te_v = y[tr_idx], y[te_idx]

proxy_vals, latency_vals = [], []
sample_front = seed0_front.sample(min(10, len(seed0_front)), random_state=0)

for _, row in sample_front.iterrows():
    params = {k.replace("param_", ""): v for k, v in row.items() if k.startswith("param_")}
    try:
        family, pipe = refit_trial_pipeline(params, X_tr_v, y_tr_v, seed=0)
        fitted_model = pipe.named_steps["clf"]
        X_te_pre = X_te_v
        for name, step in pipe.steps[:-1]:
            if name != "smote":
                X_te_pre = step.transform(X_te_pre)
        med_ms, p95_ms = measured_latency_ms(fitted_model, X_te_pre)
        proxy_vals.append(row["cost"]); latency_vals.append(med_ms)
    except Exception as e:
        print(f"  skipped one config due to: {e}")

latency_proxy_n_configs = len(proxy_vals)
if latency_proxy_n_configs >= 3:
    latency_proxy_rho, latency_proxy_pval = spearmanr(proxy_vals, latency_vals)
    print(f"Spearman(proxy, measured median latency) = {latency_proxy_rho:.3f}  (p={latency_proxy_pval:.4f}) over {latency_proxy_n_configs} configs")
else:
    latency_proxy_rho, latency_proxy_pval = None, None
    print("Not enough successful refits to compute a correlation.")

## 9. Front navigation

A Pareto front is not directly actionable — these three rules turn it into a single
recommendation under different deployment priorities.

In [38]:
def knee_point(df):
    '''Point of maximum distance from the line connecting the two normalized extremes.'''
    pts = df[["mcc", "phi", "cost"]].copy()
    pts["cost_inv"] = -pts["cost"]  # flip so all three are "higher is better"
    norm = (pts[["mcc", "phi", "cost_inv"]] - pts[["mcc", "phi", "cost_inv"]].min()) / \
           (pts[["mcc", "phi", "cost_inv"]].max() - pts[["mcc", "phi", "cost_inv"]].min() + 1e-9)
    dist_from_ideal = np.sqrt(((1 - norm) ** 2).sum(axis=1))
    return df.loc[dist_from_ideal.idxmin()]

def cost_constrained_best(df, cost_budget):
    feasible = df[df["cost"] <= cost_budget]
    if feasible.empty:
        return None
    return feasible.loc[feasible["mcc"].idxmax()]

def weighted_scalarization(df, w_mcc=0.5, w_phi=0.3, w_cost=0.2):
    '''MES, repurposed here as a front-navigation aid rather than the search objective itself.'''
    norm = df[["mcc", "phi"]].copy()
    norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-9)
    cost_norm = 1 - (df["cost"] - df["cost"].min()) / (df["cost"].max() - df["cost"].min() + 1e-9)
    mes = w_mcc * norm["mcc"] + w_phi * norm["phi"] + w_cost * cost_norm
    return df.loc[mes.idxmax()]

knee = knee_point(seed0_front)
cost_budget = seed0_front["cost"].median()
cost_pick = cost_constrained_best(seed0_front, cost_budget)
mes_pick = weighted_scalarization(seed0_front)

nav_table = pd.DataFrame([
    {"rule": "Knee point", "model_family": knee["model_family"], "mcc": knee["mcc"], "phi": knee["phi"], "cost": knee["cost"]},
    {"rule": f"Cost-constrained (budget={cost_budget:.0f})", "model_family": cost_pick["model_family"], "mcc": cost_pick["mcc"], "phi": cost_pick["phi"], "cost": cost_pick["cost"]},
    {"rule": "Weighted MES", "model_family": mes_pick["model_family"], "mcc": mes_pick["mcc"], "phi": mes_pick["phi"], "cost": mes_pick["cost"]},
])
nav_table


,rule,model_family,mcc,phi,cost
0,Knee point,dtree,0.860821,0.537570,34.2
1,Cost-constrained (budget=16),nb,0.875896,0.334575,16.0
2,Weighted MES,mlp,0.918380,0.543432,775.0


## 10. Baselines

Same folds, same seeds, equal wall-clock budget per seed. Single-objective baselines return
one point; their hypervolume is computed as a degenerate single-solution front — this is the
comparison the whole paper rests on.

In [39]:
BASELINE_TIME_BUDGET_SEC = 300  # 5 minutes, matched across FLAML / AutoGluon

def run_flaml_baseline(X, y, seed):
    from flaml import AutoML
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    automl = AutoML()
    automl.fit(X[tr_idx], y[tr_idx], task="classification", time_budget=BASELINE_TIME_BUDGET_SEC,
               metric="macro_f1" if n_classes > 2 else "roc_auc", verbose=0, seed=seed)
    pred = automl.predict(X[te_idx])
    mcc = matthews_corrcoef(y[te_idx], pred)
    return {"method": "FLAML", "seed": seed, "mcc": mcc, "phi": None, "cost": None}


def run_autogluon_baseline(X, y, seed):
    from autogluon.tabular import TabularPredictor
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    train_df = pd.DataFrame(X[tr_idx], columns=feature_names); train_df["target"] = y[tr_idx]
    test_df = pd.DataFrame(X[te_idx], columns=feature_names)
    predictor = TabularPredictor(label="target", verbosity=0,
                                  path=str(BASE_DIR / f"autogluon_seed{seed}")).fit(
        train_df, time_limit=BASELINE_TIME_BUDGET_SEC, presets="medium_quality")
    pred = predictor.predict(test_df)
    mcc = matthews_corrcoef(y[te_idx], pred)
    return {"method": "AutoGluon", "seed": seed, "mcc": mcc, "phi": None, "cost": None}


def run_random_search_3obj(X, y, seed, n_trials):
    study = optuna.create_study(
        study_name=f"moxec_random_seed{seed}", directions=["maximize", "maximize", "minimize"],
        sampler=RandomSampler(seed=seed), storage=STUDY_DB, load_if_exists=True,
    )
    remaining = max(0, n_trials - len(study.trials))
    if remaining > 0:
        study.optimize(make_objective(X, y, seed), n_trials=remaining, show_progress_bar=False)
    return study


def run_tpe_mcc_only(X, y, seed, n_trials):
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)

    def objective(trial):
        pre_steps = build_preprocessing_steps(trial, X.shape[1])
        family, model = suggest_model(trial, n_classes, random_state=seed)
        mccs = []
        for tr_idx, te_idx in skf.split(X, y):
            pipe = ImbPipeline(pre_steps + [("clf", model)]) if pre_steps else ImbPipeline([("clf", model)])
            try:
                pipe.fit(X[tr_idx], y[tr_idx])
            except Exception:
                raise optuna.TrialPruned()
            mccs.append(matthews_corrcoef(y[te_idx], pipe.predict(X[te_idx])))
        trial.set_user_attr("model_family", family)
        return float(np.mean(mccs))

    study = optuna.create_study(
        study_name=f"moxec_tpe_seed{seed}", direction="maximize",
        sampler=TPESampler(seed=seed), storage=STUDY_DB, load_if_exists=True,
    )
    remaining = max(0, n_trials - len(study.trials))
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, show_progress_bar=False)
    return study

print("Baseline functions defined.")


Baseline functions defined.


In [ ]:
# Run all baselines for every seed. This cell also resumes cleanly if interrupted.
baseline_results = []
random_studies, tpe_studies = {}, {}

for seed in range(CONFIG["n_seeds"]):
    print(f"--- seed {seed} ---")
    baseline_results.append(run_flaml_baseline(X, y, seed))
    print("  FLAML done.")

    if RUN_AUTOGLUON:
        try:
            baseline_results.append(run_autogluon_baseline(X, y, seed))
            print("  AutoGluon done.")
        except Exception as e:
            print(f"  AutoGluon skipped (error: {e})")

    random_studies[seed] = run_random_search_3obj(X, y, seed, CONFIG["n_trials"])
    print(f"  Random search (3-obj) done: {len(random_studies[seed].trials)} trials.")

    tpe_studies[seed] = run_tpe_mcc_only(X, y, seed, CONFIG["n_trials"])
    print(f"  TPE (MCC-only) done: {len(tpe_studies[seed].trials)} trials.")

baseline_df = pd.DataFrame(baseline_results)
baseline_df


--- seed 0 ---


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.6.1`. 


  FLAML done.


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch

  AutoGluon done.
  Random search (3-obj) done: 100 trials.
  TPE (MCC-only) done: 100 trials.
--- seed 1 ---


...

  AutoGluon done.
  Random search (3-obj) done: 100 trials.
  TPE (MCC-only) done: 100 trials.
--- seed 2 ---


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.6.1`. 


  FLAML done.


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch

  AutoGluon done.


## 11. Evaluation — hypervolume, per-objective breakdown, significance

Hypervolume via `pymoo`'s exact indicator (not Monte Carlo — this is the number that goes in
the paper). All three objectives are normalized to [0,1] and converted to a minimization
frame (`1 - normalized` for MCC and Φ) before computing HV, and a single shared reference
point (the nadir across every method) is used so comparisons are on equal footing.

In [ ]:
def compute_hv_for_front(front_points, global_min, global_max, ref=(1.05, 1.05, 1.05)):
    '''front_points: array of (mcc, phi, cost) tuples (cost already 'lower is better').
    Normalizes against the GLOBAL min/max across all methods being compared, then
    converts to minimization form for pymoo's HV indicator. Normalized values are
    clipped to [0,1] as a safety net -- see the note below on why a point can
    otherwise land outside the bounds and blow up the indicator.'''
    pts = np.array(front_points, dtype=float)
    mn, mx = np.array(global_min), np.array(global_max)
    norm = np.clip((pts - mn) / (mx - mn + 1e-9), 0.0, 1.0)
    # mcc, phi: higher is better -> minimize (1 - norm); cost: already lower-is-better -> minimize norm directly
    F = np.column_stack([1 - norm[:, 0], 1 - norm[:, 1], norm[:, 2]])
    ind = HV(ref_point=np.array(ref))
    return float(ind(F))


def get_tpe_point(seed):
    '''Refit TPE's best-MCC trial to recover phi/cost, for a fair HV comparison
    against the 3-objective methods. Returns (mcc, phi, cost, model_family).'''
    tpe_df = trials_to_dataframe(tpe_studies[seed])
    best_tpe_row = tpe_df.loc[tpe_df["mcc"].idxmax()]
    params_tpe = {k.replace("param_", ""): v for k, v in best_tpe_row.items() if k.startswith("param_")}
    skf_tmp = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf_tmp.split(X, y))
    fam_tpe, pipe_tpe = refit_trial_pipeline(params_tpe, X[tr_idx], y[tr_idx], seed)
    # Both train and test must go through the SAME fitted preprocessing steps before
    # being handed to faithfulness() -- the model was fit on preprocessed features
    # (e.g. post-SelectKBest), so passing raw X as the SHAP background silently
    # feature-mismatches against the fitted model (caught during notebook validation).
    X_te_pre_tpe = X[te_idx]
    X_tr_pre_tpe = X[tr_idx]
    for name, step in pipe_tpe.steps[:-1]:
        if name != "smote":
            X_te_pre_tpe = step.transform(X_te_pre_tpe)
            X_tr_pre_tpe = step.transform(X_tr_pre_tpe)
    phi_tpe, _, _ = faithfulness(pipe_tpe.named_steps["clf"], fam_tpe, X_tr_pre_tpe, X_te_pre_tpe[:CONFIG["faithfulness_sample_size"]])
    cost_tpe = structural_cost(pipe_tpe.named_steps["clf"], fam_tpe, len(tr_idx), X_te_pre_tpe.shape[1])
    return float(best_tpe_row["mcc"]), phi_tpe, cost_tpe, fam_tpe


# Refit each seed's TPE point ONCE up front. This used to happen inside the HV loop
# below, AFTER the global bounds were already fixed from NSGA-II + Random alone --
# which silently excluded TPE from the normalization space. Since TPE optimizes MCC
# only, it reliably finds a slightly higher raw MCC than either 3-objective search,
# so its normalized MCC landed above 1, made (1 - norm) negative, and inflated its
# hypervolume past the indicator's theoretical max (ref_point.prod() = 1.05**3 =
# ~1.158) -- confirmed on Dry Bean, where TPE's HV came out ~1.45-1.48. The protocol
# itself specifies the reference point as "the nadir of the union of all methods'
# solutions" (PALE_lean_protocol.md §7.1) -- TPE's point has to be part of that union.
tpe_points = {seed: get_tpe_point(seed) for seed in range(CONFIG["n_seeds"])}

# Build the global normalization bounds from every method's evaluated points (not just
# fronts) -- NSGA-II, Random, AND TPE -- so all hypervolumes are computed in the same
# objective space.
all_mcc, all_phi, all_cost = [], [], []
for seed in range(CONFIG["n_seeds"]):
    df_s = trials_to_dataframe(nsga2_studies[seed])
    all_mcc += df_s["mcc"].tolist(); all_phi += df_s["phi"].tolist(); all_cost += df_s["cost"].tolist()
    df_r = trials_to_dataframe(random_studies[seed])
    all_mcc += df_r["mcc"].tolist(); all_phi += df_r["phi"].tolist(); all_cost += df_r["cost"].tolist()
    mcc_t, phi_t, cost_t, _fam_t = tpe_points[seed]
    all_mcc.append(mcc_t); all_phi.append(phi_t); all_cost.append(cost_t)

global_min = (min(all_mcc), min(all_phi), min(all_cost))
global_max = (max(all_mcc), max(all_phi), max(all_cost))
print(f"Global bounds — MCC: [{global_min[0]:.3f}, {global_max[0]:.3f}]  "
      f"Phi: [{global_min[1]:.3f}, {global_max[1]:.3f}]  Cost: [{global_min[2]:.1f}, {global_max[2]:.1f}]")

hv_results = []
for seed in range(CONFIG["n_seeds"]):
    nsga2_front = pareto_front_df(nsga2_studies[seed])[["mcc", "phi", "cost"]].values
    hv_nsga2 = compute_hv_for_front(nsga2_front, global_min, global_max)

    random_front = pareto_front_df(random_studies[seed])[["mcc", "phi", "cost"]].values
    hv_random = compute_hv_for_front(random_front, global_min, global_max)

    hv_tpe = compute_hv_for_front([list(tpe_points[seed][:3])], global_min, global_max)

    hv_results.append({"seed": seed, "MOXEC (NSGA-II)": hv_nsga2, "Random Search (3-obj)": hv_random, "TPE (MCC-only)": hv_tpe})

hv_df = pd.DataFrame(hv_results)
hv_df

In [ ]:
# Wilcoxon signed-rank test across seeds: MOXEC vs. each baseline.
# NOTE: with a single dataset, Friedman+Nemenyi (which needs 15-30+ datasets for power)
# does not apply here. This paired seed-level test is the correct substitute for a
# single-dataset notebook; aggregate the per-dataset HV numbers this notebook saves
# across all 12 datasets to run the full Friedman+Nemenyi comparison in a follow-up notebook.
print("Wilcoxon signed-rank test (MOXEC vs. baseline), paired across seeds:\n")
wilcoxon_results = {}
for baseline in ["Random Search (3-obj)", "TPE (MCC-only)"]:
    moxec_mean = float(hv_df["MOXEC (NSGA-II)"].mean())
    baseline_mean = float(hv_df[baseline].mean())
    if hv_df["seed"].nunique() >= 3:
        stat, p = wilcoxon(hv_df["MOXEC (NSGA-II)"], hv_df[baseline])
        print(f"  MOXEC vs. {baseline}: W={stat:.3f}, p={p:.4f}  "
              f"(mean HV {moxec_mean:.4f} vs {baseline_mean:.4f})")
        wilcoxon_results[baseline] = {
            "statistic": float(stat), "p_value": float(p),
            "moxec_mean_hv": moxec_mean, "baseline_mean_hv": baseline_mean,
            "n_seeds": int(hv_df["seed"].nunique()),
        }
    else:
        print(f"  MOXEC vs. {baseline}: need >=3 seeds for Wilcoxon — currently {hv_df['seed'].nunique()}. "
              f"Reporting means only: {moxec_mean:.4f} vs {baseline_mean:.4f}")
        wilcoxon_results[baseline] = {
            "statistic": None, "p_value": None,
            "moxec_mean_hv": moxec_mean, "baseline_mean_hv": baseline_mean,
            "n_seeds": int(hv_df["seed"].nunique()),
        }

## 12. Save all artifacts

In [ ]:
# Save per-seed trial/front data for EVERY seed, not just seed 0. Seed 0 alone was
# fine for the inline plots/navigation above (one representative seed is enough to look
# at), but it silently blocked two things: (a) checking whether the family-dominance and
# front-shape patterns from seed 0 actually hold across seeds, and (b) comparing MOXEC's
# best MCC against FLAML/AutoGluon on the SAME seed rather than eyeballing across
# different seeds' summary numbers. Both matter more once you're aggregating 12 datasets.
per_seed_files = []
best_mcc_by_seed = []

for seed in range(CONFIG["n_seeds"]):
    df_s = trials_to_dataframe(nsga2_studies[seed])
    front_s = pareto_front_df(nsga2_studies[seed])

    trials_path = BASE_DIR / f"all_trials_seed{seed}.parquet"
    front_path = BASE_DIR / f"pareto_front_seed{seed}.parquet"
    df_s.to_parquet(trials_path)
    front_s.to_parquet(front_path)
    per_seed_files += [trials_path.name, front_path.name]

    moxec_best_row = df_s.loc[df_s["mcc"].idxmax()]
    row = {
        "seed": seed,
        "moxec_best_mcc": float(moxec_best_row["mcc"]),
        "moxec_best_mcc_family": moxec_best_row["model_family"],
        "n_trials": len(df_s),
        "n_pareto_front": len(front_s),
    }
    for _, brow in baseline_df[baseline_df["seed"] == seed].iterrows():
        row[f"{brow['method'].lower()}_mcc"] = float(brow["mcc"]) if pd.notna(brow["mcc"]) else None
    best_mcc_by_seed.append(row)

best_mcc_df = pd.DataFrame(best_mcc_by_seed)
best_mcc_df.to_csv(BASE_DIR / "best_mcc_by_seed.csv", index=False)

# Every number the protocol calls out as needing to be quoted directly in the paper
# (faithfulness sanity check §3, latency proxy validation §2, Wilcoxon significance
# tests §7) used to only exist as a print statement -- gone the moment a cell was
# re-run, and invisible to the cross-dataset aggregate notebook either way, since that
# only reads summary.json. All of it goes in here now.
artifact_summary = {
    "config": CONFIG,
    "dataset_meta": {"name": meta.name, "n_instances": int(X.shape[0]), "n_features": int(X.shape[1]),
                      "n_classes": int(n_classes)},
    "hypervolume_by_seed": hv_df.to_dict(orient="records"),
    "hv_normalization_bounds": {
        "mcc": [global_min[0], global_max[0]],
        "phi": [global_min[1], global_max[1]],
        "cost": [global_min[2], global_max[2]],
    },
    "tpe_points_by_seed": {
        str(seed): {"mcc": pt[0], "phi": pt[1], "cost": pt[2], "model_family": pt[3]}
        for seed, pt in tpe_points.items()
    },
    "wilcoxon_tests": wilcoxon_results,
    "faithfulness_sanity_check": {
        "spearman_rho": faithfulness_sanity_rho,
        "p_value": faithfulness_sanity_pval,
        "n_configs": faithfulness_sanity_n_configs,
    },
    "latency_proxy_validation": {
        "spearman_rho": latency_proxy_rho,
        "p_value": latency_proxy_pval,
        "n_configs": latency_proxy_n_configs,
    },
    "baseline_point_results": baseline_df.to_dict(orient="records"),
    "best_mcc_by_seed": best_mcc_df.to_dict(orient="records"),
    "navigation_picks": nav_table.to_dict(orient="records"),
}

with open(BASE_DIR / "summary.json", "w") as f:
    json.dump(artifact_summary, f, indent=2, default=str)

hv_df.to_csv(BASE_DIR / "hypervolume_by_seed.csv", index=False)
baseline_df.to_csv(BASE_DIR / "baseline_results.csv", index=False)

print(f"All artifacts saved under: {BASE_DIR.resolve()}")
print(f"Per-seed trial/front files written for all {CONFIG['n_seeds']} seeds: {per_seed_files}")
print("\nBest MCC by seed (MOXEC vs. baselines, same seed):")
print(best_mcc_df.to_string(index=False))
print("\nAll files:")
for p in sorted(BASE_DIR.glob("*")):
    print(f"  {p.name}")

---

## Next steps

- Change only the `CONFIG` cell (§1) to `uci_id` for any of the other 11 datasets in the
  MOXEC portfolio and re-run top-to-bottom — every downstream cell adapts automatically
  (binary/multiclass handling included via `task_type`).
- After running all 12 datasets, aggregate each `summary.json` / `hypervolume_by_seed.csv`
  into one table and run Friedman + Nemenyi across datasets (not just Wilcoxon across seeds)
  — see `PALE_lean_protocol.md` §7 for the exact procedure and the critical-difference diagram.
- The `RUN_AUTOGLUON` flag and `BASELINE_TIME_BUDGET_SEC` are the two easiest knobs if a run
  is too slow on free-tier Colab/Kaggle — set `RUN_AUTOGLUON = False` first.
